In [205]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# copy data
df =pd.read_csv("/content/data.csv")

# remove invalid prices
df = df[df["price"] > 0]

# convert date
df["date"] = pd.to_datetime(df["date"])
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["sale_day"] = df["date"].dt.day




In [206]:
df["house_age"] = df["sale_year"] - df["yr_built"]
df["was_renovated"] = (df["yr_renovated"] > 0).astype(int)
df["renovation_age"] = np.where(df["yr_renovated"] > 0,
                                df["sale_year"] - df["yr_renovated"],
                                0)

In [207]:
df["total_rooms"] = df["bedrooms"] + df["bathrooms"]
df["sqft_per_bedroom"] = df["sqft_living"] / (df["bedrooms"] + 1)
df["bath_per_bedroom"] = df["bathrooms"] / (df["bedrooms"] + 1)
df["lot_to_living_ratio"] = df["sqft_lot"] / (df["sqft_living"] + 1)
df["basement_ratio"] = df["sqft_basement"] / (df["sqft_living"] + 1)
df["is_basement"] = (df["sqft_basement"] > 0).astype(int)
df["is_waterfront"] = (df["waterfront"] > 0).astype(int)
df["bed_bath_interaction"] = df["bedrooms"] * df["bathrooms"]
df["sqft_times_view"] = df["sqft_living"] * (df["view"] + 1)

In [208]:
df["zipcode"] = df["statezip"].str.extract(r"(\d+)", expand=False).fillna("unknown")

In [209]:
df["city_zip"] = df["city"].astype(str) + "_" + df["zipcode"].astype(str)

In [210]:
city_price_map = df.groupby("city")["price"].median()
df["city_median_price"] = df["city"].map(city_price_map)

In [211]:
print(df["price"].describe())
print("Mean price:", df["price"].mean())
print("Median price:", df["price"].median())

count    4.551000e+03
mean     5.579059e+05
std      5.639299e+05
min      7.800000e+03
25%      3.262643e+05
50%      4.650000e+05
75%      6.575000e+05
max      2.659000e+07
Name: price, dtype: float64
Mean price: 557905.8991379443
Median price: 465000.0


In [212]:
lower = df["price"].quantile(0.05)
upper = df["price"].quantile(0.95)
df2 = df[(df["price"] >= lower) & (df["price"] <= upper)]

In [213]:
# split X and y
X = df.drop("price", axis=1)
y = np.log1p(df["price"])


In [214]:
df["price_per_sqft"] = df["price"] / df["sqft_living"]

In [215]:
df["bed_bath_interaction"] = df["bedrooms"] * df["bathrooms"]
df["sqft_times_view"] = df["sqft_living"] * (df["view"] + 1)
df["sqft_times_condition"] = df["sqft_living"] * df["condition"]
df["age_times_condition"] = df["house_age"] * df["condition"]

In [216]:
df["bedrooms_per_floor"] = df["bedrooms"] / (df["floors"] + 1)
df["living_lot_ratio"] = df["sqft_living"] / (df["sqft_lot"] + 1)
df["total_bath_value"] = df["bathrooms"] * df["sqft_living"]
df["bedroom_density"] = df["bedrooms"] / (df["sqft_living"] + 1)
df["bathroom_density"] = df["bathrooms"] / (df["sqft_living"] + 1)

In [217]:

# column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns


In [218]:
# preprocess
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])



In [219]:
# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [220]:
from xgboost import XGBRegressor

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

In [221]:
# train
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)

# convert back
y_test_actual = np.expm1(y_test)
y_pred_actual = np.expm1(y_pred)


In [222]:
# metrics
mae = mean_absolute_error(y_test_actual, y_pred_actual)
mse = mean_squared_error(y_test_actual, y_pred_actual)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_actual, y_pred_actual)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE : 92907.60658779363
MSE : 39546047073.999985
RMSE: 198861.87938868522
R2 Score: 0.73418065993591


In [223]:
results = pd.DataFrame({
    "Actual Price": y_test_actual,
    "Predicted Price": y_pred_actual
})

In [224]:
results.to_csv("house_price_predictions.csv", index=False)

from google.colab import files
files.download("house_price_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>